# Chapter 3: Linear Regression for a housing dataset

**Note:** This is the new version of the code, using scikit-learn instead of Turi Create (which was deprecated).
If you wish to see the original code in Turi Create that appears in the first edition of the book, please check [here](https://github.com/luisguiserrano/manning/blob/master/Chapter_03_Linear_Regression/DEPRECATED_Coding_linear_regression.ipynb).

### Importing packages

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import random

### Plotting functions

In [ ]:
# 画直线：斜率 slope = m（每间房价格），y 轴截距 y_intercept = b（基础价格），公式 p̂ = mr + b
def draw_line(slope, y_intercept, color='grey', linewidth=0.7, starting=0, ending=8):
    x = np.linspace(starting, ending, 1000)
    plt.plot(x, y_intercept + slope*x, linestyle='-', color=color, linewidth=linewidth)

# 画散点：横轴房间数 r，纵轴价格 p
def plot_points(features, labels):
    X = np.array(features)
    y = np.array(labels)
    plt.scatter(X, y)
    plt.xlabel('number of rooms')
    plt.ylabel('prices')

### Defining and plotting our dataset

In [ ]:
# 表 3.2 数据：房间数 r，价格 p（单位可视为千美元，或与笔记一致用「万」需自行换算）
features = np.array([1,2,3,5,6,7])   # r：房间数
labels = np.array([155, 197, 244, 356,407,448])  # p：真实价格

print(features)
print(labels)

### Coding the tricks

- Simple trick
- Absolute trick
- Square trick

In [ ]:
# 简单技巧：四种情况用 η₁、η₂ 更新 m 和 b，对应 01.例子「简单技巧的伪代码」
def simple_trick(base_price, price_per_room, num_rooms, price):
    small_random_1 = random.random()*0.1   # η₁
    small_random_2 = random.random()*0.1   # η₂
    predicted_price = base_price + price_per_room*num_rooms  # p̂ = mr + b
    # 情况1：点在上方、y轴右侧 → m'=m+η₁, b'=b+η₂
    if price > predicted_price and num_rooms > 0:
        price_per_room += small_random_1
        base_price += small_random_2
    # 情况2：点在上方、y轴左侧 → m'=m−η₁, b'=b+η₂
    if price > predicted_price and num_rooms < 0:
        price_per_room -= small_random_1
        base_price += small_random_2
    # 情况3：点在下方、y轴右侧 → m'=m−η₁, b'=b−η₂
    if price < predicted_price and num_rooms > 0:
        price_per_room -= small_random_1
        base_price -= small_random_2
    # 情况4：点在下方、y轴左侧 → m'=m+η₁, b'=b−η₂
    if price < predicted_price and num_rooms < 0:
        price_per_room -= small_random_1
        base_price += small_random_2
    return price_per_room, base_price

In [ ]:
# 绝对技巧：点在上方 p>p̂ 则 m'=m+ηr、b'=b+η；在下方则 m'=m−ηr、b'=b−η，对应 01.例子「绝对技巧的伪代码」
def absolute_trick(base_price, price_per_room, num_rooms, price, learning_rate):
    predicted_price = base_price + price_per_room*num_rooms  # p̂ = mr + b
    if price > predicted_price:   # 点在线上方：逆/顺时针旋转 + 向上平移
        price_per_room += learning_rate*num_rooms   # m' = m + ηr
        base_price += learning_rate                 # b' = b + η
    else:                          # 点在线下方：顺/逆时针旋转 + 向下平移
        price_per_room -= learning_rate*num_rooms   # m' = m − ηr
        base_price -= learning_rate                 # b' = b − η
    return price_per_room, base_price

In [ ]:
# 平方技巧：m' = m + η·r·(p−p̂)，b' = b + η·(p−p̂)，对应 01.例子「平方技巧的伪代码」
def square_trick(base_price, price_per_room, num_rooms, price, learning_rate):
    predicted_price = base_price + price_per_room*num_rooms  # p̂ = mr + b
    price_per_room += learning_rate*num_rooms*(price-predicted_price)  # 旋转线
    base_price += learning_rate*(price-predicted_price)                # 平移线
    return price_per_room, base_price

### Running the linear regression algorithm

The linear regression algorithm consists of:
- Starting with random weights
- Iterating the square (or simple, or absolute) trick many times.

In [ ]:
import random

random.seed(0)

# 线性回归算法：随机初始 m、b → 重复「随机选点 + 用 trick 更新」→ 返回拟合的 m、b
def linear_regression(features, labels, learning_rate=0.01, epochs=1000):
    price_per_room = random.random()   # 初始斜率 m
    base_price = random.random()       # 初始截距 b
    for epoch in range(epochs):
        if True:
            draw_line(price_per_room, base_price, starting=0, ending=8)
        i = random.randint(0, len(features)-1)   # 随机选一个数据点 (r, p)
        num_rooms = features[i]
        price = labels[i]
        price_per_room, base_price = square_trick(base_price, price_per_room, num_rooms, price, learning_rate=learning_rate)
    draw_line(price_per_room, base_price, 'black', starting=0, ending=8)
    plot_points(features, labels)
    print('Price per room:', price_per_room)   # 每间房价格 m
    print('Base price:', base_price)           # 基础价格 b
    return price_per_room, base_price

plt.ylim(0,500)
linear_regression(features, labels, learning_rate=0.01, epochs=1000)

### Root mean squared error function

In [ ]:
# 均方根误差 RMSE = sqrt(MSE)，MSE = (1/n)*Σ(yi−ŷi)²，对应 02.损失函数
def rmse(labels, predictions):
    n = len(labels)
    differences = np.subtract(labels, predictions)   # yi − ŷi
    return np.sqrt(1.0/n * (np.dot(differences, differences)))

### Linear regression in Scikit-Learn

In [ ]:
# 用 sklearn 拟合同一数据：系数 = 斜率 m，截距 = b，与手写 square_trick 结果接近
from sklearn.linear_model import LinearRegression

features_reshaped = features.reshape(-1, 1)
model = LinearRegression()
model.fit(features_reshaped, labels)

print("Coefficient:", model.coef_)    # 每间房价格 m
print("Intercept:", model.intercept_)  # 基础价格 b

In [ ]:
# 预测 4 个房间的价格（表 3.2 中空缺的那格）
new_point = np.array([[4]])
predicted_label = model.predict(new_point)
print("Predicted label for feature 4:", predicted_label)